# 006 Retrieval Backbone Smoke Test

Inspect the first local-model retrieval smoke test over retrieval-allowed records. This notebook is a review surface; it should not train, score holdout, or generate troubleshooting answers.

## Stage Contract

- Load records only through the downstream-use loader with `required_use=retrieval`.
- Reject answer-generation-enabled records.
- Use local BGE embeddings and CPU FAISS for the first smoke test.
- Treat this as a self-retrieval sanity check, not a real retrieval benchmark.
- Save chart-ready CSV/JSON artifacts beside the notebook-visible charts.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True})

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / 'data' / 'eval' / 'retrieval_backbone_smoke_test'
SCRIPT = ROOT / 'scripts' / 'run_retrieval_backbone_smoke_test.py'
SUMMARY = OUT / 'retrieval_smoke_summary_bge_small_en_v15.json'
SUMMARY_MD = OUT / 'retrieval_smoke_summary_bge_small_en_v15.md'

## Regenerate Smoke Artifacts

Leave `RUN_SMOKE` false unless you are intentionally refreshing the bounded retrieval smoke test. This can take around a minute on CPU.

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    result = subprocess.run(
        [sys.executable, str(SCRIPT), '--summary'],
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)

assert SUMMARY.exists(), f'Missing retrieval smoke summary: {SUMMARY}'
assert SUMMARY_MD.exists(), f'Missing retrieval smoke markdown summary: {SUMMARY_MD}'

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary['input_counts']

## Retrieval Metrics

In [ ]:
metrics = pd.read_csv(OUT / 'retrieval_metrics_bge_small_en_v15.csv')
display(metrics)

metric_plot = metrics.set_index('profile')[['hit_at_1', 'hit_at_3', 'hit_at_5', 'mrr']]
ax = metric_plot.T.plot.bar(rot=0)
ax.set_title('Retrieval Smoke Metrics')
ax.set_xlabel('metric')
ax.set_ylabel('score')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Domain Coverage

In [ ]:
domain_counts = pd.read_csv(OUT / 'retrieval_query_domain_counts_bge_small_en_v15.csv')
display(domain_counts)

ax = domain_counts.sort_values('queries').plot.barh(x='primary_domain', y='queries', legend=False)
ax.set_title('Smoke Query Coverage by Primary Domain')
ax.set_xlabel('queries')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

domain_metrics = pd.read_csv(OUT / 'retrieval_domain_metrics_bge_small_en_v15.csv')
display(domain_metrics.pivot_table(
    index='primary_domain',
    columns='profile',
    values='hit_at_5',
    aggfunc='mean',
))

## Retrieval Samples

In [ ]:
def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

flat_predictions = read_jsonl(OUT / 'retrieval_predictions_flat_bge_small_en_v15.jsonl')
sample_rows = []
for prediction in flat_predictions[:12]:
    for item in prediction['retrieved'][:3]:
        sample_rows.append({
            'query_id': prediction['query_id'],
            'query_domain': prediction['query_primary_domain'],
            'query_title': prediction['query_title'],
            'rank': item['rank'],
            'score': item['score'],
            'retrieved_id': item['record_id'],
            'retrieved_domain': item['primary_domain'],
            'retrieved_title': item['title'],
        })

pd.DataFrame(sample_rows)

## Runtime

In [ ]:
runtime = pd.DataFrame([summary['runtime_seconds']])
display(runtime)

ax = runtime.T.plot.barh(legend=False)
ax.set_title('Retrieval Smoke Runtime')
ax.set_xlabel('seconds')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Exit Criteria

- Retrieval candidates are loaded with `required_use=retrieval`.
- `answer_generation_allowed_records` is zero.
- Local BGE embeddings are produced on CPU.
- CPU FAISS search returns expected self-retrieval hits for the bounded smoke sample.
- Metrics, per-domain metrics, prediction samples, runtime, and FAISS docmap artifacts exist.
- Next step is a non-self retrieval evaluation fixture before making quality claims.